In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Datos

https://archive.ics.uci.edu/dataset/186/wine+quality

In [ ]:
# Cargar datasets de vino tinto y blanco
url_red = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
url_white = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

df_red = pd.read_csv(url_red, sep=';')
df_white = pd.read_csv(url_white, sep=';')

# Agregar columna indicando el tipo de vino
df_red['wine_type'] = 'red'
df_white['wine_type'] = 'white'

# Unir ambos datasets
df = pd.concat([df_red, df_white], axis=0).reset_index(drop=True)

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# Graficar histogramas 
df.hist(figsize=(12, 8), bins=20, edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
# Graficar boxplots 
df.plot(kind='box', figsize=(12, 8), vert=True, subplots=True, layout=(4, 4), sharex=False, sharey=False)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('quality').size()/df.shape[0]

In [ ]:
df = df[df['quality'].isin([5, 6, 7])].reset_index(drop=True)

# Tratamiento de los datos

In [ ]:
df_cat=df.loc[:,['wine_type']]
df_num=df.drop(columns=['wine_type','quality'])

scaler = StandardScaler()
df_num_scale = scaler.fit_transform(df_num)
df_num_scale= pd.DataFrame(df_num_scale,columns=df_num.columns.values)

df_cat=df_cat.reset_index(drop=True);df_num_scale=df_num_scale.reset_index(drop=True)
df_final=pd.concat([df_cat,df_num_scale],axis=1)

df_final = pd.get_dummies(df_final,drop_first=True)
df_final.head()

# Modelo de clasificacion

In [ ]:
# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(df_final,df['quality'],random_state = 123,test_size=0.2, stratify=df['quality'])

In [ ]:
arbol_clf = DecisionTreeClassifier(random_state=123)

arbol_clf.fit(X_train, y_train)

print("=== ESTRUCTURA DEL ÁRBOL ===")
print(f"Profundidad máxima alcanzada: {arbol_clf.get_depth()}")
print(f"Número total de nodos terminales (hojas): {arbol_clf.get_n_leaves()}")

In [ ]:
plt.figure(figsize=(20, 10))

# Extraer los nombres de las clases (calidades del vino) y convertirlos a texto
clases_vino = [str(c) for c in arbol_clf.classes_]

# Dibujar el árbol 
plot_tree(arbol_clf,
          feature_names=X_train.columns,
          class_names=clases_vino,
          filled=True,        
          rounded=True,       
          fontsize=10,
          max_depth=5)        

plt.title('Árbol de Decisión - Calidad del Vino', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
y_pred = arbol_clf.predict(X_test)

# Imprimir las métricas generales (Precisión, Exhaustividad, F1-Score)
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, zero_division=0))

# 3. Graficar la Matriz de Confusión
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, 
                                        cmap='Blues', 
                                        ax=ax, 
                                        colorbar=False)

plt.title('Matriz de Confusión - Predicción de Calidad del Vino', fontsize=14, fontweight='bold')
plt.xlabel('Calidad Predicha')
plt.ylabel('Calidad Real')
plt.show()

In [ ]:
arbol_clf_ajustado = DecisionTreeClassifier(
    max_depth=3,              # Límite máximo de niveles (profundidad)
    min_samples_split=20,     # Mínimo de muestras requeridas para dividir un nodo interno
    min_samples_leaf=10,      # Mínimo de muestras que deben quedar en una hoja terminal
    random_state=123
)

# Entrenar el modelo con los datos de entrenamiento
arbol_clf_ajustado.fit(X_train, y_train)

# Extraer información de la nueva estructura
print("=== ESTRUCTURA DEL ÁRBOL PODADO ===")
print(f"Profundidad máxima alcanzada: {arbol_clf_ajustado.get_depth()}")
print(f"Número total de nodos terminales (hojas): {arbol_clf_ajustado.get_n_leaves()}")

In [ ]:
plt.figure(figsize=(20, 10))

# Extraer los nombres de las clases (calidades del vino) y convertirlos a texto
clases_vino = [str(c) for c in arbol_clf.classes_]

# Dibujar el árbol 
plot_tree(arbol_clf_ajustado,
          feature_names=X_train.columns,
          class_names=clases_vino,
          filled=True,        
          rounded=True,       
          fontsize=10,
          max_depth=3)        

plt.title('Árbol de Decisión - Calidad del Vino', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
y_pred = arbol_clf_ajustado.predict(X_test)

# Imprimir las métricas generales (Precisión, Exhaustividad, F1-Score)
print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, zero_division=0))

# 3. Graficar la Matriz de Confusión
fig, ax = plt.subplots(figsize=(10, 8))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, 
                                        cmap='Blues', 
                                        ax=ax, 
                                        colorbar=False)

plt.title('Matriz de Confusión - Predicción de Calidad del Vino', fontsize=14, fontweight='bold')
plt.xlabel('Calidad Predicha')
plt.ylabel('Calidad Real')
plt.show()